### INITIAL TURBULENCE-STATE STATISTICS AFTER MITgcm SPIN-UP
## Metrics:
###   - EKE
###   - Re_eddy
###   - MLD from GLORYS
###   - bulk Richardson number


In [26]:
from pathlib import Path
import json
import numpy as np
import xarray as xr

In [27]:
# ============================================================
# SETTINGS
# ============================================================

NB_DIR = Path.cwd()

case_name = "run_domain_characterization2"              # edit
glorys_case = "GPGP_aug2020_3D_21.5_24.5_141.6_138.4"   # edit
RUN_TITLE_INFO = "run_domain_characterization"          # edit

MITGCM_FILE = (NB_DIR / f"../data/input/{case_name}.nc").resolve()
GLORYS_FILE = (NB_DIR / f"../../OGCM/data/input/{glorys_case}.nc").resolve()

# Time indices
MITGCM_TIME_INDEX = 0       # use first time entry in post-spinup MITgcm file
GLORYS_TIME_INDEX = 0       # GLORYS time used for MLD

# MLD
USE_GLORYS_MLD = True
MLD_VALUE_M = 20.0          # only used if USE_GLORYS_MLD = False

# Eddy-scale Reynolds number settings
L_EDDY_M = None             # if None, uses 0.25 * domain width
NU_EFF = 50.0               # [m2/s] effective horizontal viscosity

# Linear EOS constants
RHO0 = 1035.0               # [kg/m3]
ALPHA_T = 2.0e-4            # [1/K]
BETA_S = 7.4e-4             # [1/psu]
G = 9.81                    # [m/s2]

# Minimum shear to avoid infinite Richardson numbers
MIN_DELTA_U2 = 1e-10

# Output
OUT_DIR = (NB_DIR / f"../results/{case_name}").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ATTRS_FILE = OUT_DIR / f"{case_name}_domain_characterization.json"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def center_u_to_tracer(u):
    """
    Center MITgcm U from Xp1 to X.
    Input shape : (Z, Y, Xp1)
    Output shape: (Z, Y, X)
    """
    return 0.5 * (u[:, :, :-1] + u[:, :, 1:])


def center_v_to_tracer(v):
    """
    Center MITgcm V from Yp1 to Y.
    Input shape : (Z, Yp1, X)
    Output shape: (Z, Y, X)
    """
    return 0.5 * (v[:, :-1, :] + v[:, 1:, :])


def crop_to_common(*arrays):
    """
    Crop arrays to common trailing horizontal shape.
    Assumes the final two dimensions are Y, X.
    """
    ny = min(a.shape[-2] for a in arrays)
    nx = min(a.shape[-1] for a in arrays)
    return tuple(a[..., :ny, :nx] for a in arrays)


def positive_down_depth_from_Z(ds):
    """
    MITgcm Z is usually negative downward.
    Returns positive-down depth vector [m].
    """
    z = np.asarray(ds["Z"].values, dtype=float)

    if np.nanmean(z) < 0:
        z = -z

    return z


def interp_3d_to_depth(field_zyx, z_m, target_depth_m):
    """
    Interpolate a 3D field with shape (Z, Y, X) to a scalar depth.
    z_m must be positive downward.
    Returns field at target depth with shape (Y, X).
    """
    field_zyx = np.asarray(field_zyx, dtype=float)
    z_m = np.asarray(z_m, dtype=float)

    if target_depth_m <= z_m[0]:
        return field_zyx[0, :, :]

    if target_depth_m >= z_m[-1]:
        return field_zyx[-1, :, :]

    k2 = int(np.searchsorted(z_m, target_depth_m))
    k1 = k2 - 1

    z1 = z_m[k1]
    z2 = z_m[k2]

    w = (target_depth_m - z1) / (z2 - z1)

    return (1.0 - w) * field_zyx[k1, :, :] + w * field_zyx[k2, :, :]


def density_linear(temp, salt):
    """
    Linear EOS used only for density differences.
    Reference T/S constants cancel in Delta rho, so they are omitted.
    """
    return RHO0 * (1.0 - ALPHA_T * temp + BETA_S * salt)


def get_time_value(ds, dim_name, index):
    """
    Return raw coordinate value and printable string for a time coordinate.
    JSON-safe return.
    """
    value = ds[dim_name].isel({dim_name: index}).values

    if np.issubdtype(np.asarray(value).dtype, np.datetime64):
        value_str = np.datetime_as_string(value, unit="D")
        value_raw = value_str
    else:
        value_raw = float(np.asarray(value))
        value_str = f"{value_raw:g}"

    return value_raw, value_str


def finite_fraction(a):
    a = np.asarray(a)
    return np.isfinite(a).sum() / a.size


def print_field_qc(name, a, units=""):
    """
    Print basic finite fraction and percentile checks.
    """
    a = np.asarray(a, dtype=float)
    finite = a[np.isfinite(a)]

    print(f"\nQC: {name}")
    print(f"  shape              : {a.shape}")
    print(f"  finite fraction    : {finite_fraction(a):.4f}")

    if finite.size == 0:
        print("  WARNING: no finite values")
        return

    p = np.nanpercentile(finite, [0, 1, 5, 25, 50, 75, 95, 99, 100])
    labels = ["min", "p01", "p05", "p25", "p50", "p75", "p95", "p99", "max"]

    for lab, val in zip(labels, p):
        print(f"  {lab:>4s}              : {val:.6e} {units}")


def print_fraction_check(name, condition):
    condition = np.asarray(condition)
    frac = np.nanmean(condition)
    print(f"  fraction {name:<35s}: {frac:.4f}")


# ============================================================
# OPEN DATA
# ============================================================

print(f"Opening MITgcm file:\n{MITGCM_FILE}")
ds = xr.open_dataset(MITGCM_FILE)

print(f"\nOpening GLORYS file:\n{GLORYS_FILE}")
ds_glorys = xr.open_dataset(GLORYS_FILE)

mitgcm_T_raw, mitgcm_T_str = get_time_value(ds, "T", MITGCM_TIME_INDEX)
glorys_time_raw, glorys_time_str = get_time_value(ds_glorys, "time", GLORYS_TIME_INDEX)

print("\nSelected MITgcm post-spinup snapshot:")
print(f"  selected index : {MITGCM_TIME_INDEX}")
print(f"  T[{MITGCM_TIME_INDEX}]         : {mitgcm_T_str}")
print(f"  T units        : {ds['T'].attrs.get('units', 'not specified')}")

print("\nSelected GLORYS MLD snapshot:")
print(f"  selected index : {GLORYS_TIME_INDEX}")
print(f"  time[{GLORYS_TIME_INDEX}]      : {glorys_time_str}")


# ============================================================
# EXTRACT MITgcm FIELDS
# ============================================================

u_raw = ds["U"].isel(T=MITGCM_TIME_INDEX).values        # (Z, Y, Xp1)
v_raw = ds["V"].isel(T=MITGCM_TIME_INDEX).values        # (Z, Yp1, X)
temp = ds["Temp"].isel(T=MITGCM_TIME_INDEX).values      # (Z, Y, X)
salt = ds["S"].isel(T=MITGCM_TIME_INDEX).values         # (Z, Y, X)

u = center_u_to_tracer(u_raw)
v = center_v_to_tracer(v_raw)

u, v, temp, salt = crop_to_common(u, v, temp, salt)

z_m = positive_down_depth_from_Z(ds)

print("\nMITgcm field shapes after centering/cropping:")
print(f"  U centered : {u.shape}")
print(f"  V centered : {v.shape}")
print(f"  Temp       : {temp.shape}")
print(f"  S          : {salt.shape}")
print(f"  Z levels   : {len(z_m)}")
print(f"  depth range: {np.nanmin(z_m):.2f} to {np.nanmax(z_m):.2f} m")


# ============================================================
# QUALITY CHECKS: RAW FIELDS
# ============================================================

print("\n" + "=" * 60)
print("QUALITY CHECKS: RAW POST-SPINUP FIELDS")
print("=" * 60)

print_field_qc("surface U", u[0, :, :], "m/s")
print_field_qc("surface V", v[0, :, :], "m/s")
print_field_qc("surface speed", np.sqrt(u[0, :, :]**2 + v[0, :, :]**2), "m/s")
print_field_qc("surface Temp", temp[0, :, :], "")
print_field_qc("surface S", salt[0, :, :], "")

print("\nQC: grid / domain")
dx_est = float(np.nanmedian(np.diff(ds["X"].values)))
dy_est = float(np.nanmedian(np.diff(ds["Y"].values)))
domain_width_m = float(np.nanmax(ds["X"].values) - np.nanmin(ds["X"].values))
domain_height_m = float(np.nanmax(ds["Y"].values) - np.nanmin(ds["Y"].values))

print(f"  dx median          : {dx_est:.2f} m")
print(f"  dy median          : {dy_est:.2f} m")
print(f"  domain width       : {domain_width_m:.2f} m")
print(f"  domain height      : {domain_height_m:.2f} m")
print(f"  dx/dy ratio        : {dx_est / dy_est:.4f}")


# ============================================================
# 1. POST-SPINUP SURFACE EKE
# ============================================================

u_surf = u[0, :, :]
v_surf = v[0, :, :]

u_prime = u_surf - np.nanmean(u_surf)
v_prime = v_surf - np.nanmean(v_surf)

eke_field = 0.5 * (u_prime**2 + v_prime**2)

eke_mean = float(np.nanmean(eke_field))
eke_median = float(np.nanmedian(eke_field))
eke_std = float(np.nanstd(eke_field))

u_eddy = float(np.sqrt(2.0 * eke_mean))

print("\n" + "=" * 60)
print("1. POST-SPINUP SURFACE EKE")
print("=" * 60)
print(f"  mean U surface       : {np.nanmean(u_surf):.6e} m/s")
print(f"  mean V surface       : {np.nanmean(v_surf):.6e} m/s")
print(f"  EKE mean             : {eke_mean:.6e} m2/s2")
print(f"  EKE median           : {eke_median:.6e} m2/s2")
print(f"  EKE std              : {eke_std:.6e} m2/s2")
print(f"  U_eddy=sqrt(2EKE)    : {u_eddy:.6e} m/s")

print_field_qc("EKE field", eke_field, "m2/s2")
print_fraction_check("EKE < 0", eke_field < 0)


# ============================================================
# 2. EFFECTIVE EDDY-SCALE REYNOLDS NUMBER
# ============================================================

if L_EDDY_M is None:
    L_eddy_m = 0.25 * domain_width_m
else:
    L_eddy_m = float(L_EDDY_M)

re_eddy = float(u_eddy * L_eddy_m / NU_EFF)

print("\n" + "=" * 60)
print("2. EFFECTIVE EDDY-SCALE REYNOLDS NUMBER")
print("=" * 60)
print(f"  domain width         : {domain_width_m:.2f} m")
print(f"  L_eddy               : {L_eddy_m:.2f} m")
print(f"  L_eddy/domain width  : {L_eddy_m / domain_width_m:.4f}")
print(f"  nu_eff               : {NU_EFF:.6e} m2/s")
print(f"  Re_eddy              : {re_eddy:.6e}")

if NU_EFF <= 0:
    print("  WARNING: NU_EFF must be positive.")
if re_eddy < 10:
    print("  WARNING: Re_eddy is very low for a turbulence-regime descriptor.")
elif re_eddy < 100:
    print("  NOTE: Re_eddy is modest and strongly controlled by nu_eff.")
else:
    print("  NOTE: Re_eddy is an effective model Reynolds number, not a molecular ocean Reynolds number.")


# ============================================================
# 3. MLD FROM GLORYS
# ============================================================

if USE_GLORYS_MLD:
    mld_field = ds_glorys["mlotst"].isel(time=GLORYS_TIME_INDEX).values
    mld_vals = mld_field[np.isfinite(mld_field)]

    mld_mean = float(np.nanmean(mld_vals))
    mld_median = float(np.nanmedian(mld_vals))
    mld_std = float(np.nanstd(mld_vals))

    mld_source = str(GLORYS_FILE)
    mld_for_ri = mld_median

else:
    mld_field = np.full((1, 1), float(MLD_VALUE_M))
    mld_mean = float(MLD_VALUE_M)
    mld_median = float(MLD_VALUE_M)
    mld_std = np.nan

    mld_source = "scalar_user_input"
    mld_for_ri = float(MLD_VALUE_M)

print("\n" + "=" * 60)
print("3. MIXED-LAYER DEPTH")
print("=" * 60)
print(f"  MLD source           : {mld_source}")
print(f"  MLD mean             : {mld_mean:.2f} m")
print(f"  MLD median used      : {mld_for_ri:.2f} m")
print(f"  MLD std              : {mld_std:.2f} m")

print_field_qc("GLORYS MLD field", mld_field, "m")

print("\nQC: MLD vs MITgcm vertical grid")
print(f"  MLD used             : {mld_for_ri:.2f} m")
print(f"  shallowest MITgcm z  : {np.nanmin(z_m):.2f} m")
print(f"  deepest MITgcm z     : {np.nanmax(z_m):.2f} m")

if mld_for_ri < np.nanmin(z_m):
    print("  WARNING: MLD is shallower than the first MITgcm depth level; interpolation uses surface level.")
elif mld_for_ri > np.nanmax(z_m):
    print("  WARNING: MLD deeper than available MITgcm output; interpolation uses deepest available level.")
else:
    k_near = int(np.argmin(np.abs(z_m - mld_for_ri)))
    print(f"  nearest MITgcm level : k={k_near}, z={z_m[k_near]:.2f} m")


# ============================================================
# 4. BULK RICHARDSON NUMBER OVER MLD
# ============================================================

rho = density_linear(temp, salt)

rho_surf = rho[0, :, :]
u_surf = u[0, :, :]
v_surf = v[0, :, :]

rho_mld = interp_3d_to_depth(rho, z_m, mld_for_ri)
u_mld = interp_3d_to_depth(u, z_m, mld_for_ri)
v_mld = interp_3d_to_depth(v, z_m, mld_for_ri)

rho_surf, rho_mld, u_surf, v_surf, u_mld, v_mld = crop_to_common(
    rho_surf, rho_mld, u_surf, v_surf, u_mld, v_mld
)

delta_rho = rho_mld - rho_surf

delta_u2 = (
    (u_surf - u_mld) ** 2
    + (v_surf - v_mld) ** 2
)

delta_u = np.sqrt(delta_u2)
delta_u2_safe = np.where(delta_u2 < MIN_DELTA_U2, np.nan, delta_u2)

N2_bulk = G * delta_rho / (RHO0 * mld_for_ri)
rib_field = G * delta_rho * mld_for_ri / (RHO0 * delta_u2_safe)

rib_median = float(np.nanmedian(rib_field))
rib_mean = float(np.nanmean(rib_field))
rib_p25 = float(np.nanpercentile(rib_field, 25))
rib_p75 = float(np.nanpercentile(rib_field, 75))

print("\n" + "=" * 60)
print("4. BULK RICHARDSON NUMBER")
print("=" * 60)
print(f"  Ri_b median          : {rib_median:.6e}")
print(f"  Ri_b mean            : {rib_mean:.6e}")
print(f"  Ri_b p25             : {rib_p25:.6e}")
print(f"  Ri_b p75             : {rib_p75:.6e}")

print("\nQC: density and shear terms used in Ri_b")
print_field_qc("rho surface", rho_surf, "kg/m3")
print_field_qc("rho at MLD", rho_mld, "kg/m3")
print_field_qc("delta rho = rho_MLD - rho_surface", delta_rho, "kg/m3")
print_field_qc("delta U between surface and MLD", delta_u, "m/s")
print_field_qc("bulk N2", N2_bulk, "s-2")
print_field_qc("Ri_b field", rib_field, "")

print("\nQC: Ri_b sanity fractions")
print_fraction_check("delta_rho < 0", delta_rho < 0)
print_fraction_check("delta_u2 < MIN_DELTA_U2", delta_u2 < MIN_DELTA_U2)
print_fraction_check("N2_bulk < 0", N2_bulk < 0)
print_fraction_check("Ri_b < 0", rib_field < 0)
print_fraction_check("Ri_b < 0.25", rib_field < 0.25)
print_fraction_check("Ri_b > 10", rib_field > 10)
print_fraction_check("Ri_b is finite", np.isfinite(rib_field))

if np.nanmedian(delta_rho) <= 0:
    print("  WARNING: median delta_rho <= 0. Check density calculation or vertical coordinate.")
if np.nanmedian(N2_bulk) <= 0:
    print("  WARNING: median N2_bulk <= 0. The bulk stratification is unstable or sign convention is wrong.")
if np.nanmean(delta_u2 < MIN_DELTA_U2) > 0.05:
    print("  WARNING: more than 5% of grid points have near-zero shear; Ri_b mean may be inflated.")


# ============================================================
# STORE ATTRS JSON
# ============================================================

attrs = {
    "case_name": case_name,
    "run_title_info": RUN_TITLE_INFO,

    "mitgcm_file": str(MITGCM_FILE),
    "glorys_file": str(GLORYS_FILE),

    "mitgcm_time_index_used": int(MITGCM_TIME_INDEX),
    "mitgcm_T_value_used": mitgcm_T_raw,
    "mitgcm_T_units": ds["T"].attrs.get("units", "not specified"),

    "glorys_time_index_used": int(GLORYS_TIME_INDEX),
    "glorys_time_value_used": glorys_time_raw,

    "statistics_description": {
        "EKE": (
            "Domain-mean post-spinup surface EKE after removing the spatial "
            "domain-mean surface velocity. This includes all resolved MITgcm "
            "scales unless the input field has been filtered."
        ),
        "Re_eddy": (
            "Effective eddy-scale Reynolds number using "
            "U_eddy=sqrt(2*EKE), L_eddy, and nu_eff."
        ),
        "MLD": (
            "Domain mean/median/std of GLORYS mlotst. Median MLD is used "
            "as the vertical scale for Ri_b."
        ),
        "Ri_b": (
            "Bulk Richardson number between the surface and GLORYS median MLD, "
            "using post-spinup MITgcm Temp/S/U/V."
        ),
    },

    "eke_mean_m2_s2": eke_mean,
    "eke_median_m2_s2": eke_median,
    "eke_std_m2_s2": eke_std,
    "u_eddy_m_s": u_eddy,

    "domain_width_m": domain_width_m,
    "L_eddy_m": float(L_eddy_m),
    "nu_eff_m2_s": float(NU_EFF),
    "Re_eddy": re_eddy,

    "mld_mean_m": mld_mean,
    "mld_median_m": mld_median,
    "mld_std_m": None if not np.isfinite(mld_std) else float(mld_std),
    "mld_source": mld_source,

    "Ri_b_median": rib_median,
    "Ri_b_mean": rib_mean,
    "Ri_b_p25": rib_p25,
    "Ri_b_p75": rib_p75,

    "rho0_kg_m3": RHO0,
    "alpha_T_1_K": ALPHA_T,
    "beta_S_1_psu": BETA_S,
    "g_m_s2": G,
}

with open(OUT_ATTRS_FILE, "w") as f:
    json.dump(attrs, f, indent=4)

print(f"\nSaved attrs file:\n{OUT_ATTRS_FILE}")

Opening MITgcm file:
C:\Users\Jelle Gortemaker\Documents\Thesis\z.flow_postprocessing\data\input\run_domain_characterization2.nc

Opening GLORYS file:
C:\Users\Jelle Gortemaker\Documents\Thesis\OGCM\data\input\GPGP_aug2020_3D_21.5_24.5_141.6_138.4.nc

Selected MITgcm post-spinup snapshot:
  selected index : 0
  T[0]         : 0
  T units        : s

Selected GLORYS MLD snapshot:
  selected index : 0
  time[0]      : 2020-08-01

MITgcm field shapes after centering/cropping:
  U centered : (84, 512, 512)
  V centered : (84, 512, 512)
  Temp       : (84, 512, 512)
  S          : (84, 512, 512)
  Z levels   : 84
  depth range: 1.00 to 3889.25 m

QUALITY CHECKS: RAW POST-SPINUP FIELDS

QC: surface U
  shape              : (512, 512)
  finite fraction    : 1.0000
   min              : -2.068364e-01 m/s
   p01              : -1.786377e-01 m/s
   p05              : -1.255511e-01 m/s
   p25              : -5.583866e-02 m/s
   p50              : -1.088399e-02 m/s
   p75              : 5.817605e-